# Model 3 — Danceability & Rhythm-Style CNN (Ballroom)  💃

Classifies the rhythm/dance *style* of a clip (cha-cha, jive, waltz, quickstep,
rumba, samba, tango, Viennese-waltz). We map the model's confidence over the
up-tempo, strongly-pulsed styles into a **danceability** score that complements
the DSP danceability proxy in `app/features.py`.

> **Optional.** The DSP danceability estimate already works well; this makes it
> model-backed. Saved to `models_weights/model_3_danceability.pt`.

## 1 · Dataset — ISMIR 2004 Ballroom

~700 × 30 s clips across 8 ballroom styles, with tempo annotations — a classic
benchmark for tempo / rhythm tasks.

* Audio: http://mtg.upf.edu/ismir2004/contest/tempoContest/data1.tar.gz
* Beat/downbeat annotations (optional): https://github.com/CPJKU/BallroomAnnotations

Place as:
```
datasets/ballroom/<Style>/<clip>.wav
```
(folder-per-style is how the archive extracts.)

In [1]:
import os, sys, glob, random, time
import numpy as np
# make the project package importable from train_models/
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path: sys.path.insert(0, ROOT)
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import librosa
from app.vibe_model import GenreCNN, log_mel, GTZAN_GENRES, WEIGHTS_PATH, EMBED_DIM
from app.config import ANALYSIS_SR, N_MELS, MODELS_DIR
device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)

device mps


In [2]:
BALL = os.path.join(ROOT, 'datasets', 'ballroom')
SR=ANALYSIS_SR; FRAMES=128; BATCH=16; EPOCHS=300; LR=1e-3
assert os.path.isdir(BALL), f'Ballroom not found at {BALL} — see markdown.'
STYLES = sorted([d for d in os.listdir(BALL) if os.path.isdir(os.path.join(BALL,d))])
# up-tempo, strongly danceable styles get weight 1.0 in the danceability map
DANCE_W = {'ChaChaCha':1.0,'Jive':1.0,'Samba':1.0,'Quickstep':0.9,'Tango':0.7,
           'Rumba':0.6,'Waltz':0.4,'VienneseWaltz':0.5}
print('styles:', STYLES)
files=[]; labels=[]
for si,s in enumerate(STYLES):
    for fp in glob.glob(os.path.join(BALL,s,'*.wav')): files.append(fp); labels.append(si)
print(len(files),'clips')

styles: ['ChaChaCha', 'Jive', 'Quickstep', 'Rumba-American', 'Rumba-International', 'Rumba-Misc', 'Samba', 'Tango', 'VienneseWaltz', 'Waltz', 'nada']
698 clips


In [3]:
class Ball(Dataset):
    def __init__(self, idxs, train=True): self.idxs=idxs; self.train=train; self.c={}
    def __len__(self): return len(self.idxs)
    def __getitem__(self,k):
        i=self.idxs[k]; fp=files[i]
        if fp not in self.c:
            try: y,_=librosa.load(fp,sr=SR,mono=True)
            except Exception: return self.__getitem__((k+1)%len(self.idxs))
            self.c[fp]=log_mel(y,SR)
        m=self.c[fp]; T=m.shape[1]
        s=random.randint(0,max(0,T-FRAMES)) if self.train else max(0,(T-FRAMES)//2)
        w=m[:,s:s+FRAMES]
        if w.shape[1]<FRAMES: w=np.pad(w,((0,0),(0,FRAMES-w.shape[1])),mode='edge')
        return torch.from_numpy(w).float(), labels[i]
idx=np.arange(len(files)); np.random.seed(0); np.random.shuffle(idx)
nv=int(0.15*len(idx)); vd=Ball(idx[:nv].tolist(),False); td=Ball(idx[nv:].tolist(),True)
tdl=DataLoader(td,batch_size=BATCH,shuffle=True); vdl=DataLoader(vd,batch_size=BATCH)

In [4]:
model = GenreCNN(n_classes=len(STYLES)).to(device)
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
crit=nn.CrossEntropyLoss()
OUT=MODELS_DIR/'model_3_danceability.pt'; best=0.0
for ep in range(EPOCHS):
    model.train()
    for x,y in tdl:
        x=x.unsqueeze(1).to(device); y=torch.as_tensor(y).to(device)
        opt.zero_grad(); logits,_=model(x); loss=crit(logits,y); loss.backward(); opt.step()
    sched.step(); model.eval(); c=t=0
    with torch.no_grad():
        for x,y in vdl:
            x=x.unsqueeze(1).to(device); y=torch.as_tensor(y).to(device)
            logits,_=model(x); c+=(logits.argmax(1)==y).sum().item(); t+=len(y)
    acc=c/max(t,1); print(f'epoch {ep+1:02d} val_acc {acc:.3f}')
    if acc>best:
        best=acc
        dvec=np.array([DANCE_W.get(s,0.6) for s in STYLES],dtype='float32')
        torch.save({'state_dict':model.state_dict(),'styles':STYLES,
                    'dance_weights':dvec.tolist(),'val_acc':acc}, OUT)
print('best', round(best,3), '->', OUT)

epoch 01 val_acc 0.365
epoch 02 val_acc 0.365
epoch 03 val_acc 0.529
epoch 04 val_acc 0.519
epoch 05 val_acc 0.413
epoch 06 val_acc 0.433
epoch 07 val_acc 0.490
epoch 08 val_acc 0.529
epoch 09 val_acc 0.567
epoch 10 val_acc 0.519
epoch 11 val_acc 0.327
epoch 12 val_acc 0.567
epoch 13 val_acc 0.663
epoch 14 val_acc 0.635
epoch 15 val_acc 0.558
epoch 16 val_acc 0.413
epoch 17 val_acc 0.712
epoch 18 val_acc 0.683
epoch 19 val_acc 0.702
epoch 20 val_acc 0.702
epoch 21 val_acc 0.442
epoch 22 val_acc 0.529
epoch 23 val_acc 0.606
epoch 24 val_acc 0.529
epoch 25 val_acc 0.452
epoch 26 val_acc 0.548
epoch 27 val_acc 0.644
epoch 28 val_acc 0.558
epoch 29 val_acc 0.712
epoch 30 val_acc 0.538
epoch 31 val_acc 0.567
epoch 32 val_acc 0.731
epoch 33 val_acc 0.452
epoch 34 val_acc 0.615
epoch 35 val_acc 0.548
epoch 36 val_acc 0.673
epoch 37 val_acc 0.577
epoch 38 val_acc 0.663
epoch 39 val_acc 0.385
epoch 40 val_acc 0.644
epoch 41 val_acc 0.750
epoch 42 val_acc 0.548
epoch 43 val_acc 0.663
epoch 44 va

## 3 · Danceability at inference

`danceability = softmax(style_logits) · dance_weights` — a single 0–1 score
you can average with the DSP proxy in `app/features.py`. Ballroom style
accuracy for this net is typically ~0.80+.  Because the styles are tempo- and
groove-defined, the score correlates well with how 'mixable-on-the-floor' a
track feels.